# Data Management – GreenGarten Versandhandel GmbH

**Projekt:** Umsatzprognose bei GreenGarten Versandhandel GmbH (IHK-Zertifizierung Data Analyst)
**Autor:** Lars Petschke
**Rolle:** Data Management – Datenqualitätsprüfung, Bereinigung, Aufbau der analytischen Tabelle
**Quelle:** `data/raw/verkaufe.csv` (Rohdatensatz, 14.430 Zeilen × 16 Spalten)
**Ziel dieses Notebooks:** Den Rohdatensatz systematisch auf Qualitätsmängel prüfen, jeden
Mangel anhand der Daten belegen, direkt im Anschluss beheben und das Ergebnis als
reproduzierbare, analytisch nutzbare Tabelle unter `data/interim/verkaufe_clean.csv` ablegen.

Jeder Bereinigungsschritt folgt demselben Muster: **zeigen → begründen → beheben → prüfen.**
Die Originaldaten (`df_verkaufe_raw`) bleiben dabei unverändert; alle Bereinigungen laufen auf
einer Kopie (`df_verkaufe_clean`), damit sich jeder Schritt gegen das Original nachvollziehen lässt.

*Hinweis zu Pfaden: Dieses Notebook liegt unter `src/` und geht davon aus, dass es von dort aus
gestartet wird (Standardverhalten in Jupyter/PyCharm) – die Datenpfade sind deshalb relativ zum
Projekt-Root als `../data/...` angegeben.*

In [1]:
import pandas as pd
import numpy as np
import os

## Teil 1 — Explorative Erstuntersuchung des Rohdatensatzes

Bevor irgendetwas bereinigt wird, verschaffen wir uns einen vollständigen Überblick über den
Rohdatensatz: Struktur, Datentypen, Beispielwerte, fehlende Werte und die tatsächlich
vorkommenden Ausprägungen jeder Spalte. Erst aus dieser Bestandsaufnahme ergeben sich die
konkreten Qualitätsprobleme, die in Teil 2 behoben werden.

### 1. Rohdaten laden

In [2]:
path = "../data/raw/verkaufe.csv"
df_verkaufe_raw = pd.read_csv(path)

print(f"Datensatz erfolgreich geladen: {df_verkaufe_raw.shape[0]} Zeilen, {df_verkaufe_raw.shape[1]} Spalten.")

Datensatz erfolgreich geladen: 14430 Zeilen, 16 Spalten.


### 2. Struktur und Spalten im Überblick

In [3]:
print("Spaltennamen:")
print(df_verkaufe_raw.columns.tolist())

print("\nDatentypen der Spalten:")
print(df_verkaufe_raw.dtypes)

Spaltennamen:
['produkt_id', 'kategorie', 'hersteller', 'monat', 'monat_idx', 'preis_eur', 'wettbewerber_preis_eur', 'marketingbudget_eur', 'kampagne_aktiv', 'lagerbestand', 'bewertungen_durchschnitt', 'bewertungen_anzahl', 'vormonat_umsatz_eur', 'letzte_3_monate_umsatz_eur_avg', 'vorjahr_monat_umsatz_eur', 'umsatz_eur']

Datentypen der Spalten:
produkt_id                            str
kategorie                             str
hersteller                            str
monat                                 str
monat_idx                           int64
preis_eur                             str
wettbewerber_preis_eur            float64
marketingbudget_eur               float64
kampagne_aktiv                      int64
lagerbestand                          str
bewertungen_durchschnitt          float64
bewertungen_anzahl                  int64
vormonat_umsatz_eur               float64
letzte_3_monate_umsatz_eur_avg    float64
vorjahr_monat_umsatz_eur          float64
umsatz_eur            

### 3. Stichprobe: erste Zeilen

Ein Blick auf konkrete Werte zeigt bereits erste Auffälligkeiten in den Formaten
(`monat`, `preis_eur`, `lagerbestand`).

In [4]:
df_verkaufe_raw.head()

,produkt_id,kategorie,hersteller,monat,monat_idx,preis_eur,wettbewerber_preis_eur,marketingbudget_eur,kampagne_aktiv,lagerbestand,bewertungen_durchschnitt,bewertungen_anzahl,vormonat_umsatz_eur,letzte_3_monate_umsatz_eur_avg,vorjahr_monat_umsatz_eur,umsatz_eur
0,PR-0000,Saatgut,Hortica,01.2024,1,"1,50 €",1.41,89.61,0,67,3.7,39,NaN,NaN,NaN,6.12
1,PR-0000,Saatgut,Hortica,2024-02,2,1.5,1.69,88.16,1,69,3.7,39,6.12,6.12,NaN,13.64
2,PR-0000,Saatgut,Hortica,2024-03,3,1.5,1.61,117.38,1,64,3.7,39,13.64,9.88,NaN,16.75
3,PR-0000,Saatgut,Hortica,04.2024,4,1.5,1.40,99.85,0,72,3.7,39,16.75,12.17,NaN,22.68
4,PR-0000,Saatgut,Hortica,05/2024,5,1.5,1.45,101.72,0,35,3.7,39,22.68,17.69,NaN,11.98


### 4. Fehlende Werte im Rohdatensatz

In [5]:
df_verkaufe_raw.isna().sum()

produkt_id                           0
kategorie                            0
hersteller                           0
monat                                0
monat_idx                            0
preis_eur                            0
wettbewerber_preis_eur               0
marketingbudget_eur                433
kampagne_aktiv                       0
lagerbestand                         0
bewertungen_durchschnitt           576
bewertungen_anzahl                   0
vormonat_umsatz_eur                602
letzte_3_monate_umsatz_eur_avg       1
vorjahr_monat_umsatz_eur          7220
umsatz_eur                           0
dtype: int64

### 5. Eindeutige Werte je Spalte (erste 20)

Für kategoriale und formatkritische Spalten lohnt sich der Blick auf die tatsächlich
vorkommenden Ausprägungen — genau hier werden uneinheitliche Schreibweisen und Formate sichtbar,
die reine `dtype`- oder `describe()`-Auswertungen nicht zeigen würden.

In [6]:
for col in df_verkaufe_raw.columns:
    print(f"\nSpalte: {col}")
    print(df_verkaufe_raw[col].unique()[:20])


Spalte: produkt_id
<StringArray>
['PR-0000', 'PR-0001', 'PR-0002', 'PR-0003', 'PR-0004', 'PR-0005', 'PR-0006',
 'PR-0007', 'PR-0008', 'PR-0009', 'PR-0010', 'PR-0011', 'PR-0012', 'PR-0013',
 'PR-0014', 'PR-0015', 'PR-0016', 'PR-0017', 'PR-0018', 'PR-0019']
Length: 20, dtype: str

Spalte: kategorie
<StringArray>
[     'Saatgut', 'Bewaesserung', 'bewaesserung', 'Duengemittel',
      'DUENGER',    'Werkzeuge',     'Pflanzen',      'Pflanze',
     'PFLANZEN',       'Toepfe',  'Düngemittel', 'BEWAESSERUNG',
      'saatgut',      'SAATGUT',     'pflanzen',       'toepfe',
 'duengemittel',         'Saat',    'werkzeuge',    'WERKZEUGE']
Length: 20, dtype: str

Spalte: hersteller
<StringArray>
['Hortica', 'GartenStern', 'BioGruen', 'Vivero', 'Floraplan', 'Garda Tools']
Length: 6, dtype: str

Spalte: monat
<StringArray>
['01.2024', '2024-02', '2024-03', '04.2024', '05/2024', '2024-06', '2024-07',
 '2024-08', '09.2024', '2024-10', '11.2024', '12.2024', '01.2025', '2025-02',
 '03.2025', '04/2025'

### Fazit aus Teil 1: identifizierte Qualitätsprobleme

Die Untersuchung deckt sechs konkrete Probleme auf, die in Teil 2 einzeln behoben werden:

1. **`monat`** — drei unterschiedliche Datumsformate (`MM.YYYY`, `MM/YYYY`, `YYYY-MM`).
2. **`preis_eur`** — teils als Text mit Komma und Euro-Zeichen gespeichert (z. B. `"36,77 EUR"`).
3. **`lagerbestand`** — teils mit dem Zusatz `"Stueck"` versehen (z. B. `"65 Stueck"`).
4. **`kategorie`** — 24 statt 6 Schreibweisen durch Groß-/Kleinschreibung, Umlaut-Varianten und
   Abkürzungen (z. B. `"SAATGUT"`, `"Saat"`, `"Duengemittel"`, `"DUENGER"`).
5. **30 vollständig doppelte Zeilen** (identisch in allen Spalten).
6. **4 unplausible Werte** in der Zielvariable `umsatz_eur` — zwei Platzhalterwerte
   (999.999 / 750.000) und zwei negative Werte (-1 / -150).

Zusätzlich enthalten mehrere Spalten fehlende Werte mit unterschiedlicher Ursache
(strukturell, produktkonstant rekonstruierbar, oder ohne erkennbare Systematik) — deren
gezielte Behandlung ist Gegenstand von Schritt 14 in Teil 2.

## Teil 2 — Datenqualität dokumentieren und beheben

Jedes der oben identifizierten Probleme wird jetzt zunächst anhand der Daten belegt und direkt
im Anschluss behoben. Gearbeitet wird ausschließlich auf einer Kopie des Rohdatensatzes
(`df_verkaufe_clean`); `df_verkaufe_raw` bleibt für spätere Vergleiche unverändert erhalten.

In [7]:
df_verkaufe_clean = df_verkaufe_raw.copy()
print("Arbeitskopie 'df_verkaufe_clean' erstellt.")

Arbeitskopie 'df_verkaufe_clean' erstellt.


### 6. `monat`: Datumsformate vereinheitlichen

Ziel ist nicht nur eine lesbare Textdarstellung, sondern ein **echter Datums-Typ**
(`datetime64`): Nur damit funktionieren Sortierung, Zeitraum-Filter und Differenzen zwischen
Monaten direkt, ohne Text manuell zu zerlegen.

In [8]:
muster = df_verkaufe_clean["monat"].astype(str).str.replace(r"\d", "#", regex=True)
print("Gefundene Formate (Ziffern durch '#' ersetzt):")
muster.value_counts()

Gefundene Formate (Ziffern durch '#' ersetzt):


monat
##.####    4875
##/####    4791
####-##    4764
Name: count, dtype: int64

In [9]:
def monat_vereinheitlichen(wert):
    """Wandelt 'monat' unabhaengig vom Ursprungsformat in einen echten
    Datums-Typ (pandas Timestamp, jeweils der 1. des Monats) um.
    """
    text = str(wert).replace("/", ".").replace("-", ".")
    erster_teil, zweiter_teil = text.split(".")
    if len(erster_teil) == 4:           # Format war 'YYYY-MM' -> Reihenfolge bereits richtig
        jahr, monat_zahl = erster_teil, zweiter_teil
    else:                                # Format war 'MM.YYYY' oder 'MM/YYYY' -> Reihenfolge tauschen
        monat_zahl, jahr = erster_teil, zweiter_teil
    return pd.Timestamp(year=int(jahr), month=int(monat_zahl), day=1)


df_verkaufe_clean["monat"] = df_verkaufe_clean["monat"].apply(monat_vereinheitlichen)

print("Datentyp von 'monat':", df_verkaufe_clean["monat"].dtype)
print("Nicht umwandelbare Werte (NaT):", df_verkaufe_clean["monat"].isna().sum())
df_verkaufe_clean["monat"].head()

Datentyp von 'monat': datetime64[us]
Nicht umwandelbare Werte (NaT): 0


0   2024-01-01
1   2024-02-01
2   2024-03-01
3   2024-04-01
4   2024-05-01
Name: monat, dtype: datetime64[us]

> **Hinweis für spätere Leser:** CSV kennt keine Datentypen — beim Speichern wird `monat` als
> Text abgelegt (z. B. `"2024-01-01"`). Beim Wiedereinlesen deshalb unbedingt
> `pd.read_csv(..., parse_dates=["monat"])` verwenden, damit der `datetime64`-Typ erhalten bleibt.

### 7. `preis_eur`: Text → Zahl

In [10]:
mit_euro_zeichen = df_verkaufe_clean["preis_eur"].astype(str).str.contains("€")
print(f"Werte mit Euro-Zeichen und Komma: {mit_euro_zeichen.sum()} von {len(df_verkaufe_clean)}")
print("Beispiele:", df_verkaufe_clean.loc[mit_euro_zeichen, "preis_eur"].unique()[:5])


def preis_bereinigen(wert):
    """Entfernt das Euro-Zeichen, wandelt Komma in Punkt um, liefert einen float-Wert."""
    text = str(wert).replace("€", "").replace(",", ".").strip()
    return float(text)


df_verkaufe_clean["preis_eur"] = df_verkaufe_clean["preis_eur"].apply(preis_bereinigen)
df_verkaufe_clean["preis_eur"].describe()

Werte mit Euro-Zeichen und Komma: 578 von 14430
Beispiele: <StringArray>
['1,50 €', '36,77 €', '25,44 €', '33,80 €', '24,19 €']
Length: 5, dtype: str


count    14430.000000
mean        26.179419
std         23.310850
min          1.500000
25%          8.160000
50%         20.635000
75%         34.707500
max        143.500000
Name: preis_eur, dtype: float64

### 8. `lagerbestand`: Text-Zusatz entfernen

In [11]:
mit_text_zusatz = df_verkaufe_clean["lagerbestand"].astype(str).str.contains("[A-Za-z]")
print(f"Werte mit Text-Zusatz: {mit_text_zusatz.sum()} von {len(df_verkaufe_clean)}")
print("Beispiele:", df_verkaufe_clean.loc[mit_text_zusatz, "lagerbestand"].unique()[:5])

df_verkaufe_clean["lagerbestand"] = (
    df_verkaufe_clean["lagerbestand"]
    .astype(str)
    .str.replace("Stueck", "", regex=False)
    .str.strip()
    .astype(int)
)
df_verkaufe_clean["lagerbestand"].describe()

Werte mit Text-Zusatz: 288 von 14430
Beispiele: <StringArray>
['91 Stueck', '45 Stueck', '154 Stueck', '36 Stueck', '50 Stueck']
Length: 5, dtype: str


count    14430.000000
mean        74.391961
std         43.546379
min          1.000000
25%         42.000000
50%         66.000000
75%         97.000000
max        441.000000
Name: lagerbestand, dtype: float64

### 9. `kategorie`: Schreibweisen vereinheitlichen

24 beobachtete Schreibweisen werden über ein Mapping-Wörterbuch auf die sechs tatsächlichen
Produktkategorien reduziert. Eine Normalisierungsfunktion behandelt dabei Groß-/Kleinschreibung
und Umlaut-Varianten einheitlich, bevor das Mapping greift.

In [12]:
print(f"Urspruenglich {df_verkaufe_clean['kategorie'].nunique()} unterschiedliche Werte:")
df_verkaufe_clean["kategorie"].value_counts()

Urspruenglich 24 unterschiedliche Werte:


kategorie
Pflanzen        4218
Werkzeuge       2676
Saatgut         2496
Toepfe          1811
Duengemittel    1527
Bewaesserung    1270
PFLANZEN          47
pflanzen          46
Pflanze           44
werkzeuge         34
Werkzeug          29
WERKZEUGE         28
saatgut           26
SAATGUT           26
Saat              26
BEWAESSERUNG      24
TOEPFE            18
bewaesserung      16
Düngemittel       12
toepfe            12
duengemittel      12
Bewässerung       12
DUENGER           10
Töpfe             10
Name: count, dtype: int64

In [13]:
def kategorie_normalisieren(wert):
    """Bringt eine Kategorie-Angabe auf eine einheitliche, vergleichbare Form."""
    text = str(wert).strip().lower()
    text = (text.replace("ae", "ae").replace("ä", "ae").replace("ö", "oe")
                .replace("ü", "ue").replace("ß", "ss"))
    return text


# Zuordnung aller in den Rohdaten vorkommenden Schreibweisen zu den
# sechs tatsaechlichen Produktkategorien.
KATEGORIE_MAPPING = {
    "pflanzen": "Pflanzen", "pflanze": "Pflanzen",
    "werkzeuge": "Werkzeuge", "werkzeug": "Werkzeuge",
    "saatgut": "Saatgut", "saat": "Saatgut",
    "toepfe": "Toepfe",
    "duengemittel": "Duengemittel", "duenger": "Duengemittel",
    "bewaesserung": "Bewaesserung",
}

df_verkaufe_clean["kategorie"] = (
    df_verkaufe_clean["kategorie"].apply(kategorie_normalisieren).map(KATEGORIE_MAPPING)
)

print(f"Nach der Vereinheitlichung noch {df_verkaufe_clean['kategorie'].nunique()} Kategorien:")
print("Nicht zuordenbare Werte:", df_verkaufe_clean["kategorie"].isna().sum())
df_verkaufe_clean["kategorie"].value_counts()

Nach der Vereinheitlichung noch 6 Kategorien:
Nicht zuordenbare Werte: 0


kategorie
Pflanzen        4355
Werkzeuge       2767
Saatgut         2574
Toepfe          1851
Duengemittel    1561
Bewaesserung    1322
Name: count, dtype: int64

### 10. Vollständige Duplikate entfernen

In [14]:
anzahl_duplikate = df_verkaufe_clean.duplicated().sum()
print(f"Gefundene Duplikate: {anzahl_duplikate}")
print("Erwartung laut Struktur: 600 Produkte x 24 Monate (2024+2025) = 14.400 Zeilen; "
      f"tatsaechlich vorhanden: {len(df_verkaufe_clean)} "
      f"-> {len(df_verkaufe_clean) - 14400} zusaetzliche, doppelte Zeilen.")

df_verkaufe_clean = df_verkaufe_clean.drop_duplicates()
print(f"Zeilen nach Entfernen der Duplikate: {len(df_verkaufe_clean)}")

Gefundene Duplikate: 30
Erwartung laut Struktur: 600 Produkte x 24 Monate (2024+2025) = 14.400 Zeilen; tatsaechlich vorhanden: 14430 -> 30 zusaetzliche, doppelte Zeilen.
Zeilen nach Entfernen der Duplikate: 14400


### 11. Unplausible Werte in `umsatz_eur` markieren

Fehlerhafte Werte werden bewusst auf `NaN` gesetzt statt die Zeile zu löschen — so bleibt das
betroffene Produkt im jeweiligen Monat im Datensatz sichtbar, statt spurlos zu verschwinden.

In [15]:
print("Die beiden hoechsten Umsatzwerte:")
print(df_verkaufe_clean.nlargest(2, "umsatz_eur")[["produkt_id", "monat", "umsatz_eur"]])
print("-> Beide liegen weit ueber dem naechsthoeheren, plausiblen Wert (rund 20.600) "
      "und werden als Erfassungs-/Platzhalterfehler eingestuft.")

print("\nNegative Umsatzwerte (fachlich nicht moeglich):")
print(df_verkaufe_clean.loc[df_verkaufe_clean["umsatz_eur"] < 0,
                             ["produkt_id", "monat", "umsatz_eur"]])

unplausibel = (df_verkaufe_clean["umsatz_eur"] > 100_000) | (df_verkaufe_clean["umsatz_eur"] < 0)
print(f"\nAls fehlerhaft markierte Werte: {unplausibel.sum()}")

betroffene_zeilen = df_verkaufe_clean.loc[unplausibel, ["produkt_id", "monat"]].copy()
df_verkaufe_clean.loc[unplausibel, "umsatz_eur"] = np.nan

Die beiden hoechsten Umsatzwerte:
      produkt_id      monat  umsatz_eur
11683    PR-0486 2025-08-01    999999.0
5679     PR-0236 2025-04-01    750000.0
-> Beide liegen weit ueber dem naechsthoeheren, plausiblen Wert (rund 20.600) und werden als Erfassungs-/Platzhalterfehler eingestuft.

Negative Umsatzwerte (fachlich nicht moeglich):
      produkt_id      monat  umsatz_eur
8018     PR-0334 2024-03-01        -1.0
13086    PR-0545 2024-07-01      -150.0

Als fehlerhaft markierte Werte: 4


**Kontaminationsprüfung:** `vormonat_umsatz_eur`, `letzte_3_monate_umsatz_eur_avg` und
`vorjahr_monat_umsatz_eur` sind inhaltlich aus `umsatz_eur` abgeleitet. Für jede der vier
korrigierten Zeilen wird deshalb gezielt geprüft, ob derselbe fehlerhafte Wert auch im
Folgemonat, den beiden darauffolgenden Monaten (3-Monats-Durchschnitt) oder im Folgejahr
auftaucht.

In [16]:
kontaminierte_werte = []
for _, zeile in betroffene_zeilen.iterrows():
    pid, monat_wert = zeile["produkt_id"], zeile["monat"]
    pruefpunkte = [
        (monat_wert + pd.DateOffset(months=1), "vormonat_umsatz_eur"),
        (monat_wert + pd.DateOffset(months=1), "letzte_3_monate_umsatz_eur_avg"),
        (monat_wert + pd.DateOffset(months=2), "letzte_3_monate_umsatz_eur_avg"),
        (monat_wert + pd.DateOffset(months=3), "letzte_3_monate_umsatz_eur_avg"),
        (monat_wert + pd.DateOffset(years=1), "vorjahr_monat_umsatz_eur"),
    ]
    for ziel_monat, spalte in pruefpunkte:
        treffer = df_verkaufe_clean.loc[
            (df_verkaufe_clean["produkt_id"] == pid) & (df_verkaufe_clean["monat"] == ziel_monat),
            spalte,
        ]
        if not treffer.empty:
            wert = treffer.iloc[0]
            if pd.notna(wert) and (wert > 100_000 or wert < 0):
                kontaminierte_werte.append((pid, ziel_monat.date(), spalte, wert))

if kontaminierte_werte:
    print(f"Gefunden: {len(kontaminierte_werte)} kontaminierte Folgewerte:")
    for pid, ziel_monat, spalte, wert in kontaminierte_werte:
        print(f"  {pid} | {ziel_monat} | {spalte} = {wert}")
else:
    print(f"Keine kontaminierten Folgewerte gefunden (geprueft: {len(betroffene_zeilen)} "
          "Ausreisser x je 5 Pruefpunkte). Die abgeleiteten Spalten basieren also auf den "
          "echten, unverfaelschten Umsaetzen - keine weitere Korrektur noetig.")

Keine kontaminierten Folgewerte gefunden (geprueft: 4 Ausreisser x je 5 Pruefpunkte). Die abgeleiteten Spalten basieren also auf den echten, unverfaelschten Umsaetzen - keine weitere Korrektur noetig.


### 12. Exportfehler bei `letzte_3_monate_umsatz_eur_avg` korrigieren

Laut Projekt-Brief bedeutet `NaN` in `vormonat_umsatz_eur` den ersten Verkaufsmonat eines
Produkts (Neuprodukt ohne Historie) — für diese Zeilen kann logisch auch kein 3-Monats-
Durchschnitt existieren. Die Prüfung zeigt jedoch einen Export-Fehler: Bei 599 der 600
betroffenen Zeilen ist `letzte_3_monate_umsatz_eur_avg` trotzdem befüllt.

In [17]:
erster_monat = df_verkaufe_clean["vormonat_umsatz_eur"].isna()
fehlerhaft = erster_monat & df_verkaufe_clean["letzte_3_monate_umsatz_eur_avg"].notna()
print(f"Erster Monat je Produkt (kein Vormonat vorhanden): {erster_monat.sum()} Zeilen")
print(f"Davon mit faelschlich befuelltem 3-Monats-Durchschnitt: {fehlerhaft.sum()}")

df_verkaufe_clean.loc[erster_monat, "letzte_3_monate_umsatz_eur_avg"] = np.nan

print("Nach der Korrektur fehlend (erwartet: identisch zu 'vormonat_umsatz_eur'):",
      df_verkaufe_clean["letzte_3_monate_umsatz_eur_avg"].isna().sum())

Erster Monat je Produkt (kein Vormonat vorhanden): 600 Zeilen
Davon mit faelschlich befuelltem 3-Monats-Durchschnitt: 599
Nach der Korrektur fehlend (erwartet: identisch zu 'vormonat_umsatz_eur'): 600


### 13. Spalte `jahr` ableiten und Spaltenreihenfolge anpassen

Eine eigene Jahres-Spalte erlaubt Auswertungen wie "alle Umsätze eines Jahres" oder "alle
Umsätze eines Kalendermonats über beide Jahre hinweg", ohne jedes Mal den Text von `monat`
zerlegen zu müssen.

In [18]:
df_verkaufe_clean["jahr"] = df_verkaufe_clean["monat"].dt.year

spalten_reihenfolge = [
    "produkt_id", "kategorie", "hersteller",
    "monat", "jahr", "monat_idx",
    "preis_eur", "wettbewerber_preis_eur", "marketingbudget_eur",
    "kampagne_aktiv", "lagerbestand",
    "bewertungen_durchschnitt", "bewertungen_anzahl",
    "vormonat_umsatz_eur", "letzte_3_monate_umsatz_eur_avg",
    "vorjahr_monat_umsatz_eur", "umsatz_eur",
]
df_verkaufe_clean = df_verkaufe_clean[spalten_reihenfolge]

print("Eindeutige Jahre:", sorted(df_verkaufe_clean["jahr"].unique().tolist()))
print("\nGesamtumsatz je Jahr:")
print(df_verkaufe_clean.groupby("jahr")["umsatz_eur"].sum())
print("\nGesamtumsatz je Kalendermonat (beide Jahre zusammengefasst):")
print(df_verkaufe_clean.groupby("monat_idx")["umsatz_eur"].sum())

df_verkaufe_clean[["monat", "jahr", "monat_idx"]].head(3)

Eindeutige Jahre: [2024, 2025]

Gesamtumsatz je Jahr:
jahr
2024    3707702.59
2025    3738214.22
Name: umsatz_eur, dtype: float64

Gesamtumsatz je Kalendermonat (beide Jahre zusammengefasst):
monat_idx
1     319331.83
2     377030.81
3     678747.30
4     888069.15
5     929236.49
6     883392.36
7     796666.14
8     716608.79
9     587456.51
10    510073.88
11    392439.02
12    366864.53
Name: umsatz_eur, dtype: float64


,monat,jahr,monat_idx
0,2024-01-01,2024,1
1,2024-02-01,2024,2
2,2024-03-01,2024,3


### 14. Fehlende Werte gezielt behandeln (inkl. Flags)

Die verbleibenden fehlenden Werte werden **nicht pauschal** mit Mittelwert oder Median
aufgefüllt, sondern je Spalte einzeln nach ihrer Ursache behandelt:

| Spalte | Ursache | Strategie |
|---|---|---|
| `vormonat_umsatz_eur`, `vorjahr_monat_umsatz_eur` | strukturell (kein Vormonat/-jahr vorhanden) | `NaN` behalten + Flag |
| `bewertungen_durchschnitt` | produktkonstant, nur an einzelnen Monaten nicht erfasst | aus anderen Monaten desselben Produkts rekonstruieren |
| `marketingbudget_eur` | keine erkennbare Systematik | Median je Kategorie (robust gegen Rechtsschiefe) |

Jede dieser drei Strategien wird durch eine eigene Flag-Spalte transparent gemacht, damit
nachvollziehbar bleibt, welche Werte original und welche ersetzt bzw. rekonstruiert sind.

**14a — Strukturell fehlende Zeitreihen-Werte:** Ein Mittelwert- oder Median-Ersatz würde hier
eine nicht existierende Historie vortäuschen. Die Werte bleiben deshalb `NaN`, ergänzt um zwei
binäre Flags.

In [19]:
df_verkaufe_clean["ist_neuprodukt"] = df_verkaufe_clean["vormonat_umsatz_eur"].isna().astype(int)
df_verkaufe_clean["hat_vorjahreswert"] = df_verkaufe_clean["vorjahr_monat_umsatz_eur"].notna().astype(int)

print("'ist_neuprodukt' = 1 bei:", df_verkaufe_clean["ist_neuprodukt"].sum(),
      "Zeilen (erwartet: 600, ein erster Monat je Produkt)")
print("'hat_vorjahreswert' = 0 bei:", (df_verkaufe_clean["hat_vorjahreswert"] == 0).sum(),
      "Zeilen (erwartet: 7.200, alle Zeilen aus 2024)")

'ist_neuprodukt' = 1 bei: 600 Zeilen (erwartet: 600, ein erster Monat je Produkt)
'hat_vorjahreswert' = 0 bei: 7200 Zeilen (erwartet: 7.200, alle Zeilen aus 2024)


**14b — `bewertungen_durchschnitt`:** Prüfung zeigt, dass der Wert je Produkt über alle Monate
konstant ist — fehlende Monatswerte lassen sich daher exakt aus anderen Monaten desselben
Produkts rekonstruieren (Forward-/Backward-Fill je `produkt_id`), statt sie zu schätzen.

In [20]:
eindeutige_werte_je_produkt = df_verkaufe_clean.groupby("produkt_id")["bewertungen_durchschnitt"].nunique()
print("Produkte mit genau einem eindeutigen (nicht-NaN) Bewertungswert:",
      f"{(eindeutige_werte_je_produkt == 1).sum()} von {df_verkaufe_clean['produkt_id'].nunique()}")

df_verkaufe_clean["bewertungen_ergaenzt"] = df_verkaufe_clean["bewertungen_durchschnitt"].isna().astype(int)
print("Fehlend vor Rekonstruktion:", df_verkaufe_clean["bewertungen_ergaenzt"].sum())

df_verkaufe_clean["bewertungen_durchschnitt"] = (
    df_verkaufe_clean.groupby("produkt_id")["bewertungen_durchschnitt"]
    .transform(lambda s: s.ffill().bfill())
)
print("Verbleibend fehlend nach Rekonstruktion:",
      df_verkaufe_clean["bewertungen_durchschnitt"].isna().sum())

Produkte mit genau einem eindeutigen (nicht-NaN) Bewertungswert: 600 von 600
Fehlend vor Rekonstruktion: 576
Verbleibend fehlend nach Rekonstruktion: 0


**14c — `marketingbudget_eur`:** Weder Kategorie noch aktive Kampagne erklären das Fehlen der
Werte. Die leicht rechtsschiefe Verteilung wird über den robusteren Median je Kategorie ersetzt.

In [21]:
fehlend_marketing = df_verkaufe_clean["marketingbudget_eur"].isna().sum()
print(f"'marketingbudget_eur' fehlend: {fehlend_marketing} Zeilen "
      f"({fehlend_marketing / len(df_verkaufe_clean):.1%})")

median_je_kategorie = df_verkaufe_clean.groupby("kategorie")["marketingbudget_eur"].median()
print("Median je Kategorie (globaler Median: "
      f"{df_verkaufe_clean['marketingbudget_eur'].median():.2f} Euro, Schiefe: "
      f"{df_verkaufe_clean['marketingbudget_eur'].skew():.2f} -> Median statt Mittelwert):")
print(median_je_kategorie.round(2))

df_verkaufe_clean["marketingbudget_geschaetzt"] = df_verkaufe_clean["marketingbudget_eur"].isna().astype(int)
df_verkaufe_clean["marketingbudget_eur"] = df_verkaufe_clean["marketingbudget_eur"].fillna(
    df_verkaufe_clean["kategorie"].map(median_je_kategorie)
)
print("Verbleibend fehlend nach Median-Ersatz:",
      df_verkaufe_clean["marketingbudget_eur"].isna().sum())

'marketingbudget_eur' fehlend: 432 Zeilen (3.0%)
Median je Kategorie (globaler Median: 88.09 Euro, Schiefe: 0.35 -> Median statt Mittelwert):
kategorie
Bewaesserung    88.53
Duengemittel    90.72
Pflanzen        87.79
Saatgut         85.58
Toepfe          88.08
Werkzeuge       89.11
Name: marketingbudget_eur, dtype: float64
Verbleibend fehlend nach Median-Ersatz: 0


**14d — Flag-Spalten neben ihre Quellspalte einsortieren:**

In [22]:
spalten_reihenfolge_mit_flags = [
    "produkt_id", "kategorie", "hersteller",
    "monat", "jahr", "monat_idx",
    "preis_eur", "wettbewerber_preis_eur",
    "marketingbudget_eur", "marketingbudget_geschaetzt",
    "kampagne_aktiv", "lagerbestand",
    "bewertungen_durchschnitt", "bewertungen_ergaenzt", "bewertungen_anzahl",
    "vormonat_umsatz_eur", "ist_neuprodukt",
    "letzte_3_monate_umsatz_eur_avg",
    "vorjahr_monat_umsatz_eur", "hat_vorjahreswert",
    "umsatz_eur",
]
df_verkaufe_clean = df_verkaufe_clean[spalten_reihenfolge_mit_flags]

print(f"Datensatz jetzt: {df_verkaufe_clean.shape[0]} Zeilen, {df_verkaufe_clean.shape[1]} Spalten.")
print("Verbleibende NaN insgesamt je Spalte (nur Spalten mit NaN):")
verbleibend = df_verkaufe_clean.isna().sum()
verbleibend[verbleibend > 0]

Datensatz jetzt: 14400 Zeilen, 21 Spalten.
Verbleibende NaN insgesamt je Spalte (nur Spalten mit NaN):


vormonat_umsatz_eur                600
letzte_3_monate_umsatz_eur_avg     600
vorjahr_monat_umsatz_eur          7200
umsatz_eur                           4
dtype: int64

### 15. Bereinigten Datensatz speichern

Die drei verbliebenen NaN-Gruppen (`vormonat_umsatz_eur`, `letzte_3_monate_umsatz_eur_avg`,
`vorjahr_monat_umsatz_eur`) sind bewusst so belassen — sie sind strukturell begründet und über
die Flags `ist_neuprodukt` / `hat_vorjahreswert` transparent gemacht.

In [23]:
os.makedirs("../data/interim", exist_ok=True)

INTERIM_PATH = "../data/interim/verkaufe_clean.csv"
df_verkaufe_clean.to_csv(INTERIM_PATH, index=False, date_format="%Y-%m-%d")

print(f"Bereinigter Datensatz gespeichert unter: {INTERIM_PATH}")
print(f"Finale Form: {df_verkaufe_clean.shape[0]} Zeilen, {df_verkaufe_clean.shape[1]} Spalten "
      f"(Rohdatensatz: {df_verkaufe_raw.shape[0]} Zeilen, {df_verkaufe_raw.shape[1]} Spalten).")

Bereinigter Datensatz gespeichert unter: ../data/interim/verkaufe_clean.csv
Finale Form: 14400 Zeilen, 21 Spalten (Rohdatensatz: 14430 Zeilen, 16 Spalten).


## Zusammenfassung

Aus dem Rohdatensatz (14.430 Zeilen × 16 Spalten) entsteht eine bereinigte, analytisch
nutzbare Tabelle (14.400 Zeilen × 21 Spalten: 600 Produkte × 24 Monate, 17 fachliche Spalten
plus vier Flag-Spalten). Behoben wurden: uneinheitliche Datumsformate, Text-/Zahl-Mischformate,
24 uneinheitliche Kategorie-Schreibweisen, 30 vollständige Duplikate, vier unplausible
Zielwerte sowie ein Exportfehler bei den 3-Monats-Durchschnitten. Fehlende Werte wurden je
Spalte einzeln nach Ursache behandelt statt pauschal aufgefüllt. Jeder Schritt ist oben anhand
der tatsächlichen Daten belegt und direkt geprüft — die vollständige Spaltendokumentation
inklusive Wertebereichen befindet sich in `reports/Datenbeschreibung.md`.